# Laboratorio 2 — Árboles de decisión y Random Forest (15 min)
**Curso:** Aprendizaje Automático · MCDA 2026-1  
**Sesión:** 02 — Árboles, regularización y ensambles

## Objetivo
Entrenar un árbol de decisión, **ver el overfitting en vivo**, controlarlo con regularización y compararlo contra un Random Forest.

## Dataset
**Wine** de `sklearn`: 178 vinos, 13 variables químicas, 3 clases (cultivar de origen).

## Agenda (15 min)
| Paso | Tiempo | Qué haces |
|------|--------|-----------|
| 1. Cargar + partición | 1 min | Base común |
| 2. Árbol sin restricciones | 3 min | Entrenar, visualizar, detectar overfitting |
| 3. Regularizar el árbol | 3 min | `max_depth`, `min_samples_leaf` |
| 4. Random Forest | 3 min | Comparar con el árbol |
| 5. Retos | 5 min | 3 micro-retos individuales |


## Paso 1 · Cargar datos y partir (1 min)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

data = load_wine(as_frame=True)
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
print(f'Train: {X_train.shape} · Test: {X_test.shape} · Clases: {np.unique(y)}')

## Paso 2 · Árbol sin restricciones (3 min)
Dejamos que el árbol crezca todo lo que quiera. Comparamos el desempeño en **train** vs **test** para detectar overfitting.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

tree_full = DecisionTreeClassifier(random_state=42)  # sin restricciones
tree_full.fit(X_train, y_train)

acc_train = accuracy_score(y_train, tree_full.predict(X_train))
acc_test  = accuracy_score(y_test,  tree_full.predict(X_test))
print(f'Profundidad del árbol : {tree_full.get_depth()}')
print(f'Hojas                 : {tree_full.get_n_leaves()}')
print(f'Accuracy TRAIN        : {acc_train:.3f}')
print(f'Accuracy TEST         : {acc_test:.3f}')
print(f'Gap (overfitting)     : {acc_train - acc_test:+.3f}')

In [ ]:
plt.figure(figsize=(14, 6))
plot_tree(tree_full, feature_names=X.columns, class_names=data.target_names,
          filled=True, rounded=True, fontsize=7)
plt.title('Árbol sin restricciones')
plt.show()

**Observa:** el train accuracy normalmente queda en 1.000 → el árbol memorizó los datos de entrenamiento. El test suele ser menor: eso es overfitting.

## Paso 3 · Regularizar el árbol (3 min)
Barremos `max_depth` y graficamos train vs test para ver el punto óptimo.

In [ ]:
depths = range(1, 11)
train_scores, test_scores = [], []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, m.predict(X_train)))
    test_scores.append(accuracy_score(y_test,  m.predict(X_test)))

plt.figure(figsize=(8, 4))
plt.plot(depths, train_scores, 'o-', label='Train')
plt.plot(depths, test_scores,  's-', label='Test')
plt.xlabel('max_depth'); plt.ylabel('Accuracy')
plt.title('Curva de complejidad: sesgo ↔ varianza')
plt.legend(); plt.grid(True, alpha=.3); plt.show()

**Pregunta:** ¿A partir de qué profundidad el test deja de mejorar (o empeora) mientras el train sigue subiendo?

## Paso 4 · Random Forest (3 min)
Reemplazamos el árbol por un ensamble y comparamos.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

print(f'Árbol sin restricciones — Test: {acc_test:.3f}')
print(f'Random Forest (200 árboles) — Test: {accuracy_score(y_test, rf.predict(X_test)):.3f}')

In [ ]:
# Importancia de variables según el bosque
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
imp.plot.barh(figsize=(7, 5), title='Importancia de variables (Random Forest)')
plt.tight_layout(); plt.show()

## Paso 5 · Retos individuales (5 min)

### Reto 1 · Árbol regularizado
Entrena un `DecisionTreeClassifier` con `max_depth=3` y `min_samples_leaf=5`. Reporta accuracy en train y test. ¿Se cerró el gap?

### Reto 2 · Bosque más pequeño
Entrena un Random Forest con `n_estimators=10` y con `n_estimators=500`. ¿Cuánto mejora el test al pasar de 10 a 500? ¿Vale la pena el costo?

### Reto 3 · ¿Qué variable manda?
Mira el gráfico de importancias e identifica las **3 variables más importantes** según el bosque. ¿Coinciden con las primeras divisiones del árbol sin restricciones?

In [ ]:
# Reto 1 — tu código aquí


In [ ]:
# Reto 2 — tu código aquí


In [ ]:
# Reto 3 — tu código aquí
# pista: imp.tail(3)


## Cierre
Viste que un árbol profundo memoriza (alta varianza) y que la regularización lo estabiliza. El Random Forest promedia muchos árboles sobre-ajustados y, gracias a la aleatoriedad de filas (bootstrap) y columnas, reduce esa varianza sin aumentar el sesgo. Esto es exactamente lo que motiva el trade-off sesgo-varianza que vimos en teoría.